# 05 | Statistical Baselines and Rolling-Origin Backtesting

## Study objective

This notebook evaluates whether the SOFC health trajectory contains forecastable temporal structure beyond a simple persistence assumption.

Statistical baseline models are tested using leakage-safe rolling-origin backtesting across multiple forecast horizons. Their performance establishes the minimum benchmark that more advanced machine-learning models must outperform to demonstrate genuine forecasting value.

In [ ]:
from pathlib import Path

import pandas as pd
import seaborn as sns

from sofc_health.evaluation.backtest import rolling_backtest
from sofc_health.models.baselines import (
    damped_local_trend,
    drift,
    persistence,
)
from sofc_health.models.statistical import (
    exponential_smoothing_forecast,
)
from sofc_health.validation.metrics import regression_metrics

ROOT = Path.cwd()

if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent

table = pd.read_parquet(ROOT / "data" / "processed" / "modeling_table.parquet")

print("Project root:", ROOT)
print("Modeling-table shape:", table.shape)
print("Cells:", sorted(table["cell_id"].unique()))
print(
    "Composite SOH missing values:",
    table["soh_composite_pct"].isna().sum(),
)

## How the baseline forecasts work

Before using machine learning, we first ask a simple question:

> Can a complex model predict future SOH better than a reasonable forecast based only on the cell's past behaviour?

To answer this question, we compare several statistical baseline models.

### 1. Persistence forecast

Persistence assumes that future SOH will remain equal to the latest observed SOH:

$$
\widehat{y}_{k+h}=y_k
$$

Here:

- $y_k$ is the SOH measured at the current assessment
- $h$ is the forecast horizon
- $\widehat{y}_{k+h}$ is the predicted SOH at a future assessment

Persistence is a strong baseline for slowly changing health trajectories. If SOH changes only slightly between assessments, carrying the latest observed value forward may be difficult to beat.

The assumption is:

> The most recent health measurement is the best available estimate of future health.

Persistence does not estimate a degradation rate. It simply assumes no change during the forecast horizon.

### 2. Drift forecast

The drift model assumes that the average historical rate of change will continue into the future:

$$
\widehat{y}_{k+h}
=
y_k
+
h
\left(
\frac{y_k-y_1}{k-1}
\right)
$$

Here:

- $y_1$ is the SOH at the first assessment
- $y_k$ is the SOH at the current assessment
- $k-1$ is the number of assessment intervals
- $h$ is the number of future assessments
- $\widehat{y}_{k+h}$ is the forecasted SOH

The average historical change per assessment is:

$$
b_k
=
\frac{y_k-y_1}{k-1}
$$

Therefore, the drift forecast can also be written as:

$$
\widehat{y}_{k+h}
=
y_k+h b_k
$$

If $b_k<0$, the model predicts continued degradation.

If $b_k>0$, the model predicts continued performance improvement.

The main assumption is:

> The average historical trend will remain relevant during the forecast horizon.

This assumption may fail if degradation accelerates, slows down, recovers temporarily or changes after a redox event.

### 3. Damped local-trend forecast

The drift model uses the complete history from the first assessment. A local-trend model instead estimates the recent direction of the SOH trajectory.

A damped trend gradually reduces the influence of that estimated trend as the forecast horizon increases:

$$
\widehat{y}_{k+h}
=
y_k
+
\left(
\sum_{i=1}^{h}\phi^i
\right)b_k
$$

Here:

- $b_k$ is the estimated local SOH trend
- $\phi$ is the damping parameter
- $0<\phi<1$
- $h$ is the forecast horizon

When $\phi$ is close to 1, the trend continues strongly into the future.

When $\phi$ is smaller, the projected trend weakens more quickly.

For example, if a temporary performance drop creates a steep negative local trend, damping prevents that short-term slope from being extrapolated indefinitely.

The main assumption is:

> Recent health behaviour contains useful information, but its influence becomes less reliable farther into the future.

### 4. Exponential-smoothing forecast

Exponential smoothing estimates the current SOH level by giving more importance to recent measurements and progressively less importance to older measurements.

A simplified level update is:

$$
\ell_k
=
\alpha y_k
+
(1-\alpha)\ell_{k-1}
$$

Here:

- $\ell_k$ is the updated SOH level
- $\ell_{k-1}$ is the previous estimated level
- $y_k$ is the latest measured SOH
- $\alpha$ is the smoothing parameter
- $0<\alpha<1$

A large value of $\alpha$ gives more weight to the latest observation. The model reacts quickly to recent changes but may also react strongly to noise.

A small value of $\alpha$ gives more weight to the historical level. The model becomes smoother but may respond slowly to real degradation changes.

A trend component can also be estimated:

$$
b_k
=
\beta(\ell_k-\ell_{k-1})
+
(1-\beta)b_{k-1}
$$

Here:

- $b_k$ is the updated trend
- $\beta$ controls how quickly the trend responds to new information
- $0<\beta<1$

A damped exponential-smoothing forecast can be written as:

$$
\widehat{y}_{k+h}
=
\ell_k
+
\left(
\sum_{i=1}^{h}\phi^i
\right)b_k
$$

Exponential smoothing is useful when the health trajectory contains a slowly changing level and trend.

## Rolling-origin evaluation

The baseline models must be evaluated using only information available before each prediction.

For a prediction made at assessment $k$, the model may use:

$$
y_1,y_2,\ldots,y_k
$$

It must not use:

$$
y_{k+1},y_{k+2},\ldots
$$

After making the prediction, the evaluation origin moves forward and the model receives one or more additional observations.

For example:

| Forecast origin | Training history | Predicted assessment |
|---|---|---|
| $k=8$ | Assessments 1 to 8 | Assessment 11 |
| $k=9$ | Assessments 1 to 9 | Assessment 12 |
| $k=10$ | Assessments 1 to 10 | Assessment 13 |

For a three-assessment horizon:

$$
h=3
$$

This process simulates how the model would operate in practice while preventing future-information leakage.

## How forecast performance is evaluated

### Mean Absolute Error

Mean Absolute Error measures the average absolute distance between the measured and predicted SOH:

$$
\mathrm{MAE}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left|
y_i-\widehat{y}_i
\right|
$$

MAE is reported in SOH percentage points.

For example, an MAE of 1.5 means that the model's predictions differ from measured SOH by an average of 1.5 percentage points.

MAE treats all errors in direct proportion to their magnitude.

### Root Mean Squared Error

Root Mean Squared Error gives greater weight to large prediction errors:

$$
\mathrm{RMSE}
=
\sqrt{
\frac{1}{n}
\sum_{i=1}^{n}
\left(
y_i-\widehat{y}_i
\right)^2
}
$$

Because errors are squared, one severe prediction failure can increase RMSE substantially.

If RMSE is much larger than MAE, the model probably makes a small number of unusually large errors.

### Signed bias

Using the following convention:

$$
\mathrm{Bias}
=
\frac{1}{n}
\sum_{i=1}^{n}
\left(
\widehat{y}_i-y_i
\right)
$$

Interpretation:

- $\mathrm{Bias}>0$ means the model systematically predicts SOH too high
- $\mathrm{Bias}<0$ means the model systematically predicts SOH too low
- $\mathrm{Bias}\approx0$ means there is little average directional error

Bias close to zero does not guarantee accurate predictions because positive and negative errors may cancel.

For health forecasting, positive bias can be especially risky because it may make a degraded cell appear healthier than it actually is.

### Mean Absolute Scaled Error

Mean Absolute Scaled Error compares the model's MAE with the typical one-step change observed in the training history:

$$
\mathrm{MASE}
=
\frac{
\frac{1}{n}
\sum_{i=1}^{n}
\left|
y_i-\widehat{y}_i
\right|
}{
\frac{1}{T-1}
\sum_{t=2}^{T}
\left|
y_t-y_{t-1}
\right|
}
$$

The denominator represents the average absolute change between consecutive training observations.

Interpretation:

- $\mathrm{MASE}<1$ means the model performs better than the naive in-sample scale
- $\mathrm{MASE}=1$ means the model performs similarly to that naive scale
- $\mathrm{MASE}>1$ means the model performs worse than that naive scale

MASE is useful because it is scale-independent and allows comparisons across cells with different SOH variability.

## Model-selection principle

A more complex model is useful only if it improves performance on unseen data.

The evaluation should therefore ask:

1. Does the model beat persistence?
2. Does it perform well at more than one forecast horizon?
3. Does it avoid systematic positive bias?
4. Does it work across different cells?
5. Does it remain stable for regular and randomized-redox cells?
6. Is the improvement large enough to matter operationally?

A high training score is not sufficient. The model must demonstrate reliable forecasting improvement under chronological and cell-level validation.

In [ ]:
models = {
    "persistence": persistence,
    "drift": drift,
    "damped_trend": damped_local_trend,
    "ets": exponential_smoothing_forecast,
}

forecast_horizons = [1, 3, 5, 10]
predictions = []

for model_name, model_function in models.items():
    for forecast_horizon in forecast_horizons:
        result = rolling_backtest(
            table,
            model_function,
            target="soh_composite_pct",
            min_history=8,
            horizon=forecast_horizon,
        )

        # Keep only the requested final horizon.
        result = result[result["horizon"] == forecast_horizon].copy()

        result["model"] = model_name
        predictions.append(result)

predictions = pd.concat(
    predictions,
    ignore_index=True,
)

print("Prediction rows:", len(predictions))
print(
    "Models:",
    sorted(predictions["model"].unique()),
)
print(
    "Horizons:",
    sorted(predictions["horizon"].unique()),
)

cell_metrics = (
    predictions.groupby(["model", "horizon", "cell_id"])
    .apply(
        lambda group: pd.Series(
            regression_metrics(
                group["y_true"].to_numpy(),
                group["y_pred"].to_numpy(),
            )
        ),
        include_groups=False,
    )
    .reset_index()
)

display(cell_metrics.head(12).round(3))

macro_metrics = (
    cell_metrics.groupby(["model", "horizon"])[["mae", "rmse", "bias", "r2"]].mean().reset_index()
)

macro_metrics.round(3)

In [ ]:
persistence_mae = cell_metrics.loc[
    cell_metrics["model"] == "persistence",
    ["cell_id", "horizon", "mae"],
].rename(
    columns={
        "mae": "persistence_mae",
    }
)

skill_table = cell_metrics.merge(
    persistence_mae,
    on=["cell_id", "horizon"],
    how="left",
)

skill_table["mae_skill"] = 1 - skill_table["mae"] / skill_table["persistence_mae"]

skill_table["regime"] = skill_table["cell_id"].apply(
    lambda cell: "randomized" if cell.startswith("R") else "regular"
)

display(
    skill_table[
        [
            "model",
            "horizon",
            "cell_id",
            "regime",
            "mae",
            "persistence_mae",
            "mae_skill",
        ]
    ]
    .sort_values(["horizon", "model", "cell_id"])
    .head(20)
    .round(3)
)

In [ ]:
skill_summary = (
    skill_table[skill_table["model"] != "persistence"]
    .groupby(["model", "horizon", "regime"])
    .agg(
        median_skill=("mae_skill", "median"),
        mean_skill=("mae_skill", "mean"),
        minimum_skill=("mae_skill", "min"),
        maximum_skill=("mae_skill", "max"),
        cells_better_than_persistence=(
            "mae_skill",
            lambda values: int((values > 0).sum()),
        ),
        cell_count=("cell_id", "nunique"),
    )
    .reset_index()
)

skill_summary.round(3)

In [ ]:
import matplotlib.pyplot as plt

models_to_plot = ["damped_trend", "drift", "ets"]
cell_order = ["N1", "N2", "N3", "N4", "N5", "N6", "R1", "R2"]
horizon_order = [1, 3, 5, 10]

fig, axes = plt.subplots(
    1,
    len(models_to_plot),
    figsize=(17, 7),
    sharey=True,
)

for ax, model in zip(axes, models_to_plot, strict=True):
    heatmap_data = (
        skill_table.loc[skill_table["model"] == model]
        .pivot(
            index="cell_id",
            columns="horizon",
            values="mae_skill",
        )
        .reindex(index=cell_order, columns=horizon_order)
    )

    sns.heatmap(
        heatmap_data,
        annot=True,
        fmt=".2f",
        cmap="RdBu_r",
        center=0,
        robust=True,
        linewidths=0.5,
        cbar=model == models_to_plot[-1],
        ax=ax,
    )

    ax.set_title(model.replace("_", " ").title())
    ax.set_xlabel("Forecast horizon")
    ax.set_ylabel("Cell" if ax is axes[0] else "")

fig.suptitle(
    "MAE skill relative to persistence\nPositive values indicate improvement",
    fontsize=16,
    y=1.02,
)

plt.tight_layout()
plt.show()

## Conclusion: classical forecasting baselines

Persistence is the strongest overall benchmark for forecasting composite SOH. Across all eight cells, none of the classical forecasting models consistently reduced MAE relative to simply carrying the most recent SOH value forward.

The prediction problem becomes more difficult as the horizon increases. Persistence remains competitive because SOH changes gradually over many regular-cell assessments. Its weakness is positive forecast bias: when SOH is declining, the last observed value tends to overestimate future health.

The cell-level skill analysis revealed behaviour that was hidden by the aggregate metrics. ETS produced substantial long-horizon improvements for several regular-redox cells. At horizon 10, its MAE skill reached 0.69 for N1, 0.63 for N2 and 0.50 for N3. However, it did not improve every regular cell.

The drift model was less reliable. It performed well for N1 and N2 but failed severely for N3 and N4. This happens because an estimated linear degradation rate can be extrapolated too strongly when the underlying SOH trajectory is nonlinear, noisy or temporarily recovering.

All three classical models performed worse than persistence for R1 and R2 at every tested horizon. These cells exhibit stronger local SOH fluctuations and recovery behaviour, so a single local-level or trend model is unable to represent their dynamics adequately.

The forecasting evidence therefore supports three decisions:

1. Persistence must remain the mandatory benchmark for every advanced model.
2. Regular and randomized-redox cells must be reported separately.
3. A pooled model should be accepted only if leave-one-cell-out validation demonstrates stable transfer across both regimes.

The MAE skill score is defined as:

$$
\text{Skill}_{c,h}
=
1-
\frac{\text{MAE}_{\text{model},c,h}}
{\text{MAE}_{\text{persistence},c,h}}
$$

A positive value means the model improves upon persistence. A negative value means that the additional model complexity produces a larger forecasting error.

These results do not yet establish ETS as the final forecasting model. They establish it as the strongest classical candidate to challenge with feature-based machine-learning models under leakage-safe, leave-one-cell-out validation.